In [1]:
import sys
!"{sys.executable}" -m pip install networkx numpy scipy torch scikit-learn matplotlib
# torch-geometric may require special install steps on Windows and is optional for this notebook



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import networkx as nx
import torch
import torch.nn as nn
import torch.nn.functional as F
import random

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)


In [3]:
import os
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

# Create empty graph
G = nx.Graph()

# Load Budapest edgelist as an unweighted graph
path = os.path.join("..", "Datasets", "out.foldoc")

# Each line: V1 V2 weight. We ignore the weight and use binary edges.
edges = np.loadtxt(path, dtype=int, usecols=(0, 1))

# If the file has a single edge, np.loadtxt returns a 1D array, so normalize it.
if edges.ndim == 1:
    edges = edges.reshape(1, 2)

G.add_edges_from(edges)

# Basic info
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

# Draw graph only if it is reasonably small; otherwise skip plotting.
if G.number_of_nodes() <= 200:
    pos = nx.spring_layout(G, seed=42)
    nx.draw(G, pos, with_labels=True, node_size=50, font_size=8)
    plt.show()
else:
    print("Graph is large; skipping full plot.")

Nodes: 13356
Edges: 91471
Graph is large; skipping full plot.


In [4]:
# adjacency matrix
nodelist = list(G.nodes())
A = nx.to_numpy_array(G, nodelist=nodelist)
print(A)

[[0. 1. 1. ... 0. 0. 0.]
 [1. 0. 0. ... 0. 0. 0.]
 [1. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 1.]
 [0. 0. 0. ... 0. 1. 0.]]


In [5]:
#Degree
deg = np.array([G.degree(n) for n in nodelist])
deg

array([ 99, 199, 179, ...,  14,  14,   7], shape=(13356,))

In [6]:
import time

n = len(nodelist)
dist_matrix = np.zeros((n, n))
t_start = time.time()

print(f"Populating shortest path matrix row-by-row for {n} nodes safely...")

for i, u in enumerate(nodelist):
    # Compute shortest path lengths from node u only (uses virtually 0 extra RAM)
    lengths = nx.single_source_shortest_path_length(G, u)
    
    for j, v in enumerate(nodelist):
        if i != j and v in lengths:
            dist_matrix[i, j] = lengths[v]
            
    # Print progress so you know it's working
    if (i + 1) % 1000 == 0 or (i + 1) == n:
        elapsed = time.time() - t_start
        print(f"Progress: {i + 1}/{n} nodes processed | Time elapsed: {elapsed:.1f}s")

print("\nDone! Shortest path matrix constructed successfully.")
print(dist_matrix)


Populating shortest path matrix row-by-row for 13356 nodes safely...
Progress: 1000/13356 nodes processed | Time elapsed: 24.4s
Progress: 2000/13356 nodes processed | Time elapsed: 48.0s
Progress: 3000/13356 nodes processed | Time elapsed: 72.0s
Progress: 4000/13356 nodes processed | Time elapsed: 95.6s
Progress: 5000/13356 nodes processed | Time elapsed: 119.1s
Progress: 6000/13356 nodes processed | Time elapsed: 144.1s
Progress: 7000/13356 nodes processed | Time elapsed: 169.7s
Progress: 8000/13356 nodes processed | Time elapsed: 193.4s
Progress: 9000/13356 nodes processed | Time elapsed: 217.2s
Progress: 10000/13356 nodes processed | Time elapsed: 240.5s
Progress: 11000/13356 nodes processed | Time elapsed: 265.0s
Progress: 12000/13356 nodes processed | Time elapsed: 288.5s
Progress: 13000/13356 nodes processed | Time elapsed: 312.9s
Progress: 13356/13356 nodes processed | Time elapsed: 321.4s

Done! Shortest path matrix constructed successfully.
[[0. 1. 1. ... 3. 3. 3.]
 [1. 0. 2. 

4. Node Feature Extraction (Exact Formulas)

Paper defines three measures.

NLI  = Local influence

NGI  = Global influence

NLGC = Hybrid influence

### 4.1 Global Influence (NGI)

**Paper formula:**

$$\text{NGI}_i = \sum_{i \neq j} \frac{\sqrt{d(v_j) + \alpha}}{d_{ij}}$$

**Where:**
*   $d(v_j)$ = degree of node $j$
*   $d_{ij}$ = shortest path distance between node $i$ and node $j$
*   $\alpha$ = constant parameter

### 🔹 Node Global Influence (NGI) – Description

**Definition:**
NGI is a metric used to measure the overall importance of a node by considering its interaction with all other nodes in the network.

**Core Idea:**
It combines both:
*   **Local information** → node degree
*   **Global information** → shortest path distance

**Computation:**
For each node, influence is calculated by summing contributions from all other nodes based on:
*   **Smoothed degree** of the contributing node (using square root scaling)
*   **Distance** between the two nodes

**Role of Degree ($d(v_j)$):**
Nodes with higher connections contribute more influence. However, instead of using the raw degree directly, a square root transformation is applied to moderate the dominance of high-degree nodes.

**Role of Distance ($d_{ij}$):**
Influence decreases as distance increases, ensuring closer nodes have stronger impact. The inverse relationship gives higher weight to nearby nodes.

**Role of $\alpha$ (alpha):**
*   Added inside the square root to stabilize the computation.
*   Prevents zero or very small degree values from reducing influence too much.
*   Helps in smoothing the contribution of nodes.
*   $\alpha = 0.5$ provides a balanced contribution.

**Key Advantage:**
NGI captures both local connectivity and global positioning while ensuring balanced influence using square root scaling.

**Interpretation:**
A node with a higher NGI value is more influential in the network, considering both its connectivity and its position relative to other nodes.

In [7]:
alpha = 0.5

NGI = np.zeros(n)

for i, u in enumerate(nodelist):
    for j, v in enumerate(nodelist):
        if i != j and dist_matrix[i, j] != 0:
            NGI[i] += np.sqrt(deg[j] + alpha) / dist_matrix[i, j]

print("NGI:", NGI)

NGI: [17852.07368722 17910.7286704  18461.84399651 ... 13110.84673396
 12571.98998307 12294.45128034]


### 4.2 Node Local Influence (NLI)

**Formula:**

$$NLI_i = \frac{d(v_i) \times \log_2 \left( \sum_{j \in N_i} e^{d(v_j)} \right)}{n}$$

---

### 🔹 Node Local Influence (NLI) – Description

**Definition:**
NLI is a metric used to measure the importance of a node based on its local neighborhood structure, focusing only on its immediate connections.

**Core Idea:**
It captures influence using:
*   **Node’s own degree**
*   **Contribution from its neighboring nodes**

**Computation:**
For each node, influence is calculated by:
1. Taking the degree of the node.
2. Multiplying it with the logarithm of the sum of exponential contributions from its neighbors.
3. Normalizing by the total number of nodes ($n$).

**Role of Degree ($d(v_i)$):**
The degree reflects direct connectivity; nodes with more neighbors have higher local influence.

**Role of Neighbor Contribution:**
Each neighbor contributes via an exponential function, which:
*   Amplifies the importance of highly connected neighbors.
*   Highlights strong local structures.

**Role of Logarithm ($\\log$):**
*   Compresses large values from exponential growth.
*   Prevents numerical explosion and ensures balanced scaling.

**Role of Normalization ($n$):**
Dividing by total nodes ensures values are comparable across different graph sizes.

**Key Advantage:**
NLI focuses purely on local structure, making it effective in identifying nodes that are well-connected locally and surrounded by influential neighbors.

**Interpretation:**
A node with higher NLI is more influential within its immediate neighborhood, even if it is not globally central.

In [8]:
NLI = np.zeros(n)
log2_e = np.log2(np.e)

for i, u in enumerate(nodelist):
    neighbors = list(G.neighbors(u))
    if len(neighbors) > 0:
        # Extract neighbor degrees
        neighbor_degrees = np.array([G.degree(v) for v in neighbors])
        max_deg = np.max(neighbor_degrees)
        
        # Log-Sum-Exp trick to prevent overflow
        exp_sum = np.sum(np.exp(neighbor_degrees - max_deg))
        log2_exp_sum = max_deg * log2_e + np.log2(exp_sum)
        
        NLI[i] = (G.degree(u) * log2_exp_sum) / n
    else:
        NLI[i] = 0

print("NLI:", NLI)

NLI: [ 7.78510909  8.53378529 14.07610633 ...  0.20415496  0.02827829
  0.14442073]


4.3 Hybrid Influence

Paper multiplies them.

$$\text{NLGC}_i = \text{NLI}_i \times \text{NGI}_i$$

In [9]:
NLGC = NLI * NGI
NLGC

array([138980.3411122 , 152846.31287997, 259870.87917703, ...,
         2676.6443724 ,    355.51432505,   1775.57362866], shape=(13356,))

### 🔹 Multi-Scale Feature Construction

**Definition:**
Multi-scale feature construction is used to capture node influence at different neighborhood levels by progressively aggregating information from neighboring nodes.

**Core Idea:**
Instead of relying on a single-scale measure, influence is computed across multiple levels:
*   **Level 1** → node itself
*   **Level 2** → node + immediate neighbors
*   **Level 3** → node + extended neighborhood

---

### 🧬 Computation: NLI-based Features

**Level 1:**
$$W_{NLI1}(i) = NLI_i$$

**Level 2:**
$$W_{NLI2}(i) = W_{NLI1}(i) + \sum_{j \in N(i)} W_{NLI1}(j)$$

**Level 3:**
$$W_{NLI3}(i) = W_{NLI2}(i) + \sum_{j \in N(i)} W_{NLI2}(j)$$

---

### 🌍 Computation: NGI-based Features

**Level 1:**
$$W_{NGI1}(i) = NGI_i$$

**Level 2:**
$$W_{NGI2}(i) = W_{NGI1}(i) + \sum_{j \in N(i)} W_{NGI1}(j)$$

**Level 3:**
$$W_{NGI3}(i) = W_{NGI2}(i) + \sum_{j \in N(i)} W_{NGI2}(j)$$

---

### ✅ Key Advantages
*   **Higher-Order Influence:** Captures both local and extended neighborhood importance.
*   **Rich Structural Info:** Provides a multidimensional view of a node's position for learning.
*   **Propagation Awareness:** Helps GCNs understand how influence spreads across multiple hops.

**Interpretation:**
Nodes with higher values at deeper levels (e.g., $NLI_3$, $NGI_3$) are not only locally important but are also strategically connected to other influential regions in the graph.

In [10]:
import numpy as np

nodelist = list(G.nodes())
node_index = {node: i for i, node in enumerate(nodelist)}
n = len(nodelist)

# --- NLI Multi-scale ---
W_NLI1 = NLI.copy()
W_NLI2 = np.zeros(n)
W_NLI3 = np.zeros(n)

# NLI2
for i, u in enumerate(nodelist):
    neighbor_sum = 0
    for v in G.neighbors(u):
        j = node_index[v]
        neighbor_sum += W_NLI1[j]
    W_NLI2[i] = W_NLI1[i] + neighbor_sum

# NLI3
for i, u in enumerate(nodelist):
    neighbor_sum = 0
    for v in G.neighbors(u):
        j = node_index[v]
        neighbor_sum += W_NLI2[j]
    W_NLI3[i] = W_NLI2[i] + neighbor_sum


# --- NGI Multi-scale ---
W_NGI1 = NGI.copy()
W_NGI2 = np.zeros(n)
W_NGI3 = np.zeros(n)

# NGI2
for i, u in enumerate(nodelist):
    neighbor_sum = 0
    for v in G.neighbors(u):
        j = node_index[v]
        neighbor_sum += W_NGI1[j]
    W_NGI2[i] = W_NGI1[i] + neighbor_sum

# NGI3
for i, u in enumerate(nodelist):
    neighbor_sum = 0
    for v in G.neighbors(u):
        j = node_index[v]
        neighbor_sum += W_NGI2[j]
    W_NGI3[i] = W_NGI2[i] + neighbor_sum


print("W_NLI1:", W_NLI1)
print("W_NLI2:", W_NLI2)
print("W_NLI3:", W_NLI3)

print("W_NGI1:", W_NGI1)
print("W_NGI2:", W_NGI2)
print("W_NGI3:", W_NGI3)

W_NLI1: [ 7.78510909  8.53378529 14.07610633 ...  0.20415496  0.02827829
  0.14442073]
W_NLI2: [251.61732296 235.73751986 284.5142361  ...  10.02878623   3.07656572
   8.5475316 ]
W_NLI3: [ 9139.50535365 10883.63310742 13051.40714044 ...   308.0863598
   151.91916353   105.13146206]
W_NGI1: [17852.07368722 17910.7286704  18461.84399651 ... 13110.84673396
 12571.98998307 12294.45128034]
W_NGI2: [1523132.65331406 2982208.51800387 2719265.74713998 ...  201070.33677409
  196431.79912531  105225.22163872]
W_NGI3: [70646543.2736399  82589895.76648982 95038572.34551246 ...
  4869862.64972915  2484870.44787935  3463311.74159026]


### 🔹 Neighborhood Matrix Construction

**Definition:**
A neighborhood matrix is constructed for each node to represent its local structural information using a fixed-size subgraph.

**Core Idea:**
Instead of using the entire graph, a localized neighborhood subgraph is extracted for each node, ensuring:
*   Reduced computational complexity
*   Consistent input size for learning models

---

### ⚙️ Computation Steps:
1.  **Extract** one-hop neighbors of the target node.
2.  **Rank** neighbors based on importance scores (e.g., $W_{NLI3}$ or $W_{NGI3}$).
3.  **Select** the top $L$ neighbors.
4.  **Construct** a $(L+1) 	imes (L+1)$ adjacency matrix including the node and selected neighbors.

**Role of Parameter $L$:**
*   Determines the size of the neighborhood.
*   Controls how much local information is captured.
*   Ensures uniform matrix size across all nodes.

---

### ✅ Key Advantage
*   **Efficiency:** Reduces computational complexity.
*   **Robustness:** Avoids bias from high-degree nodes.
*   **Consistency:** Provides structured and consistent input for GCN.

**Interpretation:**
Each node is represented by a fixed-size local subgraph, capturing its most important neighbors and their mutual connections.

In [11]:
import numpy as np

# choose L <= max neighbors: use a fixed neighborhood size of 40 for the large graph
L = 40

def neighborhood_matrix(node):

    nbrs = list(G.neighbors(node))

    # sort neighbors using importance (W_NLI3 or W_NGI3)
    nbrs_sorted = sorted(nbrs, key=lambda x: W_NLI3[nodelist.index(x)], reverse=True)

    nbrs_selected = nbrs_sorted[:L]

    # keep a fixed size L+1; pad with placeholder values if the node has fewer neighbors
    nodes = [node] + nbrs_selected
    if len(nodes) < L + 1:
        nodes += [None] * (L + 1 - len(nodes))

    size = L + 1
    mat = np.zeros((size, size))

    for i, u in enumerate(nodes):
        for j, v in enumerate(nodes):
            if u is not None and v is not None and G.has_edge(u, v):
                mat[i, j] = 1

    return mat, nodes


# Example: for the first node in the graph
mat0, nodes0 = neighborhood_matrix(nodelist[0])

print("Neighborhood Matrix:\n", mat0)
print("Nodes used:", nodes0)

Neighborhood Matrix:
 [[0. 1. 1. ... 1. 1. 1.]
 [1. 0. 1. ... 1. 1. 0.]
 [1. 1. 0. ... 0. 0. 0.]
 ...
 [1. 1. 0. ... 0. 0. 0.]
 [1. 1. 0. ... 0. 0. 0.]
 [1. 0. 0. ... 0. 0. 0.]]
Nodes used: [np.int64(22), np.int64(12404), np.int64(12460), np.int64(1645), np.int64(7811), np.int64(18), np.int64(10), np.int64(12739), np.int64(6847), np.int64(4979), np.int64(4689), np.int64(6257), np.int64(4638), np.int64(6857), np.int64(6767), np.int64(2744), np.int64(7603), np.int64(9654), np.int64(7595), np.int64(5442), np.int64(11806), np.int64(1723), np.int64(4644), np.int64(5558), np.int64(4980), np.int64(4681), np.int64(1099), np.int64(11970), np.int64(5795), np.int64(7477), np.int64(12715), np.int64(8146), np.int64(3846), np.int64(9943), np.int64(6278), np.int64(3137), np.int64(9834), np.int64(3847), np.int64(4986), np.int64(9833), np.int64(7054)]


### 🔹 Structural Channel Construction

**Definition:**
Structural channel construction embeds node feature information into the neighborhood matrix to generate multiple feature-aware representations of each node.

**Core Idea:**
Instead of using only structural adjacency, node features are incorporated into the matrix to create channels that capture both:
*   **Structural relationships**
*   **Node importance**

---

### ⚙️ Computation:
For each node, a neighborhood matrix is constructed and node features (e.g., NLI, NGI) are embedded into this matrix according to specific rules:

**Channel Construction Rules:**
*   **Diagonal elements:** Represent the feature value of the node itself.
*   **Off-diagonal elements:**
    *   If an edge exists → assign the feature value of the neighbor.
    *   If no edge exists → the value remains zero.

**Channels Created:**
*   **Local influence channels:** $E^{(NLI1)}$, $E^{(NLI2)}$, $E^{(NLI3)}$
*   **Global influence channels:** $E^{(NGI1)}$, $E^{(NGI2)}$, $E^{(NGI3)}$

---

### ✅ Key Advantage
*   **Integration:** Combines structural and feature information seamlessly.
*   **Power:** Enhances the representation power of nodes.
*   **Scalability:** Provides multi-scale learning capability.

**Interpretation:**
Each channel represents a feature-enriched local subgraph, enabling the model to learn both node importance and connectivity patterns simultaneously.

In [12]:
import numpy as np


def embed_channel(mat, nodes, feature_dict):

    size = mat.shape[0]
    out = np.zeros((size, size))

    for i in range(size):
        for j in range(size):

            u = nodes[i]
            v = nodes[j]

            # diagonal → self feature
            if i == j:
                out[i, j] = feature_dict.get(u, 0)

            # edge exists → take neighbor feature
            elif mat[i, j] == 1:
                out[i, j] = feature_dict.get(v, 0)

    return out

### 🔹 Purpose of Structural Channel Construction

Structural channel construction is performed to transform the graph into a format that can effectively capture both **node importance** and **local structural relationships** in a unified representation.

Graph data is inherently irregular, where each node may have a different number of neighbors. This makes it difficult to directly apply deep learning models that require fixed-size inputs.

To address this, a neighborhood matrix is first constructed for each node, ensuring a consistent structure. However, this matrix only represents connectivity and does not include any information about node importance.

Therefore, node features such as local influence (NLI) and global influence (NGI) are embedded into the neighborhood matrix to create **feature-aware channels**.

**In these channels:**
*   The **diagonal elements** represent the importance of the node itself
*   The **off-diagonal elements** represent the importance of neighboring nodes if a connection exists

This transformation allows the model to simultaneously learn:
1.  **Who is connected to whom** (structure)
2.  **How important each node is** (features)

By constructing multiple channels at different scales (NLI1–3 and NGI1–3), the model is able to capture multi-level influence propagation, improving its ability to identify key nodes.

---

### ✅ Key Benefit
This approach enables the graph to be represented as a **multi-channel matrix** (similar to images), making it suitable for deep learning models while preserving both structural and semantic information.

In [13]:
# convert arrays to dict (important)
NLI_dict = {node: NLI[i] for i, node in enumerate(nodelist)}
NGI_dict = {node: NGI[i] for i, node in enumerate(nodelist)}

# example for node 3
mat, nodes = neighborhood_matrix(3)

E_NLI1 = embed_channel(mat, nodes, NLI_dict)
E_NLI2 = embed_channel(mat, nodes, dict(zip(nodelist, W_NLI2)))
E_NLI3 = embed_channel(mat, nodes, dict(zip(nodelist, W_NLI3)))

E_NGI1 = embed_channel(mat, nodes, NGI_dict)
E_NGI2 = embed_channel(mat, nodes, dict(zip(nodelist, W_NGI2)))
E_NGI3 = embed_channel(mat, nodes, dict(zip(nodelist, W_NGI3)))

print("E_NLI1:\n", E_NLI1)
print("E_NLI2:\n", E_NLI2)
print("E_NLI3:\n", E_NLI3)
print("E_NGI1:\n", E_NGI1)
print("E_NGI2:\n", E_NGI2)
print("E_NGI3:\n", E_NGI3)

E_NLI1:
 [[3.14549862e+00 3.22413610e+01 3.22413609e+01 ... 3.45659190e-02
  3.02451837e-02 5.61696182e-02]
 [3.14549862e+00 3.22413610e+01 3.22413609e+01 ... 0.00000000e+00
  0.00000000e+00 0.00000000e+00]
 [3.14549862e+00 3.22413610e+01 3.22413609e+01 ... 0.00000000e+00
  0.00000000e+00 0.00000000e+00]
 ...
 [3.14549862e+00 0.00000000e+00 0.00000000e+00 ... 3.45659190e-02
  0.00000000e+00 0.00000000e+00]
 [3.14549862e+00 0.00000000e+00 0.00000000e+00 ... 0.00000000e+00
  3.02451837e-02 0.00000000e+00]
 [3.14549862e+00 0.00000000e+00 0.00000000e+00 ... 0.00000000e+00
  0.00000000e+00 5.61696182e-02]]
E_NLI2:
 [[ 104.37781636 1307.03331494  530.28872471 ...    5.16109866
     5.21502514    4.07937193]
 [ 104.37781636 1307.03331494  530.28872471 ...    0.
     0.            0.        ]
 [ 104.37781636 1307.03331494  530.28872471 ...    0.
     0.            0.        ]
 ...
 [ 104.37781636    0.            0.         ...    5.16109866
     0.            0.        ]
 [ 104.37781636    0.

### 🔹 Channel Tensor Construction

**Definition:**
Channel tensor construction combines multiple feature-embedded neighborhood matrices into a unified multi-dimensional representation for each node.

**Core Idea:**
For each node, six structural channels are generated by embedding different feature representations ($NLI_1$–$NLI_3$ and $NGI_1$–$NGI_3$) into the neighborhood matrix.

---

### ⚙️ Computation:
1.  A **neighborhood matrix** of size $(L+1) \times (L+1)$ is constructed.
2.  **Six feature matrices** are generated by embedding node importance values into the structural layout.
3.  These matrices are stacked to form a tensor of size:
    $$6 \times (L+1) \times (L+1)$$

---

### ✅ Key Advantage
*   **Multi-Perspective:** Captures multiple levels of node importance.
*   **Hybrid Representation:** Combines structural connectivity and feature importance.
*   **Deep Learning Ready:** Enables standard CNN or GCN models to process graph data efficiently.

**Interpretation:**
Each node is represented as a multi-channel tensor, where each channel encodes a different aspect of node influence and neighborhood structure.

In [14]:
channels = []

for node in G.nodes():
    mat, nodes = neighborhood_matrix(node)

    # create feature dicts (node → value), skipping None placeholders
    f1 = {n: NLI_dict.get(n, 0) for n in nodes}
    f2 = {n: W_NLI2[node_index[n]] if n is not None else 0 for n in nodes}
    f3 = {n: W_NLI3[node_index[n]] if n is not None else 0 for n in nodes}

    f4 = {n: NGI_dict.get(n, 0) for n in nodes}
    f5 = {n: W_NGI2[node_index[n]] if n is not None else 0 for n in nodes}
    f6 = {n: W_NGI3[node_index[n]] if n is not None else 0 for n in nodes}

    # create channels
    c1 = embed_channel(mat, nodes, f1)
    c2 = embed_channel(mat, nodes, f2)
    c3 = embed_channel(mat, nodes, f3)

    c4 = embed_channel(mat, nodes, f4)
    c5 = embed_channel(mat, nodes, f5)
    c6 = embed_channel(mat, nodes, f6)

    # stack → (6, L+1, L+1) as float32 to save 50% memory
    tensor = np.stack([c1, c2, c3, c4, c5, c6]).astype(np.float32)

    channels.append(tensor)

# final shape: (num_nodes, 6, L+1, L+1) as float32 (only takes 500 MB now!)
channels = np.array(channels, dtype=np.float32)

print(channels.shape)


(13356, 6, 41, 41)


### 🔹 Channel Attention Module

**Definition:**
The channel attention module is used to adaptively learn the importance of different feature channels and enhance the representation of informative channels.

**Core Idea:**
Not all feature channels contribute equally to node importance. Therefore, an attention mechanism is introduced to assign weights to each channel dynamically.

---

### ⚙️ Computation:
1.  **Global average pooling** is applied to each channel to obtain a compact representation.
2.  The pooled values are passed through **fully connected layers**.
3.  A **sigmoid activation** generates normalized weights between 0 and 1.
4.  These weights are **multiplied** with the input feature maps.

---

### ✅ Key Advantage
*   **Feature Selection:** Highlights important feature channels.
*   **Noise Reduction:** Suppresses less relevant information.
*   **Robustness:** Improves model robustness and generalization.

**Interpretation:**
Channels representing more meaningful structural or influence patterns receive higher weights, allowing the model to focus on the most relevant information.

In [15]:
import torch
import torch.nn as nn

class ChannelAttention(nn.Module):

    def __init__(self, channels=6, reduction=2):
        super(ChannelAttention, self).__init__()

        # Global Average Pooling
        self.avg_pool = nn.AdaptiveAvgPool2d(1)

        # Fully Connected Layers (SE block)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid()
        )

    def forward(self, x):
        # x: (batch, channels, height, width)

        b, c, _, _ = x.size()

        # Step 1: Global Average Pooling
        y = self.avg_pool(x).view(b, c)

        # Step 2: FC → channel weights
        y = self.fc(y).view(b, c, 1, 1)

        # Step 3: Multiply weights
        out = x * y

        return out

In [16]:
# test input
x = torch.randn(2, 6, 3, 3)   # batch=2, channels=6

model = ChannelAttention(6)

out = model(x)

print("Input shape:", x.shape)
print("Output shape:", out.shape)

Input shape: torch.Size([2, 6, 3, 3])
Output shape: torch.Size([2, 6, 3, 3])


### 🔹 NLGCN Model Architecture

**Definition:**
The NLGCN model is a convolutional neural network designed to learn node influence from multi-channel structural representations of graph data.

**Core Idea:**
The model processes the constructed channel tensor using convolutional layers to extract structural patterns, while a channel attention mechanism enhances important feature channels.

---

### 🏗 Architecture:
*   **Input:** Multi-channel tensor of size $6 \times (L+1) \times (L+1)$.
*   **Channel Attention:** Assigns adaptive weights to feature channels.
*   **Convolution Layer 1:** Extracts local structural patterns (followed by Batch Normalization and ReLU).
*   **Pooling Layer:** Reduces spatial dimensions and retains key features.
*   **Convolution Layer 2:** Learns higher-level structural representations.
*   **Fully Connected Layers:** Transform extracted features into the final influence score.

---

### ✅ Key Advantage
*   **Hybrid Learning:** Captures both local and multi-scale structural patterns.
*   **Attention-Driven:** Enhances feature learning using the attention mechanism.
*   **Structured Processing:** Efficiently processes graph data in a consistent matrix format.

**Interpretation:**
The model learns how node importance is influenced by both its local structure and multi-scale neighborhood features, producing a final influence score.

### 🛠 Model Component Summary

| Part | Role |
| :--- | :--- |
| **Channel Attention** | Adaptively select and weight important feature channels |
| **Convolution Layer 1** | Extract local structural patterns from the neighborhood |
| **Max Pooling** | Reduce spatial dimensions and retain significant features |
| **Convolution Layer 2** | Learn higher-order structural representations |
| **Fully Connected** | Map structural features to the final node influence score |

In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class NLGCN(nn.Module):

    def __init__(self):
        super(NLGCN, self).__init__()

        self.attention = ChannelAttention(6)

        # Conv 1
        self.conv1 = nn.Conv2d(6, 16, kernel_size=2)
        self.bn = nn.BatchNorm2d(16)
        self.pool = nn.MaxPool2d(2)

        # -------- Conv 2 (for larger graphs) --------
        # Uncomment this block when using larger graphs (L >= 4 or bigger input size)
        # self.conv2 = nn.Conv2d(16, 32, kernel_size=2)
        # self.pool2 = nn.MaxPool2d(2)

        # -------- FC Layers --------
        # For L=40, input size after conv1+pool is 16 x 20 x 20
        self.fc1 = nn.Linear(16 * 20 * 20, 8)
        self.fc2 = nn.Linear(8, 1)

        # For even larger graphs or extra conv layers, adjust this accordingly
        # self.fc1 = nn.Linear(32 * k * k, 64)  # adjust k based on output size
        # self.fc2 = nn.Linear(64, 1)

    def forward(self, x):

        # x: (batch, 6, L+1, L+1)

        x = self.attention(x)

        x = self.conv1(x)
        x = self.bn(x)
        x = F.relu(x)

        x = self.pool(x)

        # -------- Conv 2 (for larger graphs) --------
        # Uncomment when input size is large enough
        # x = self.conv2(x)
        # x = F.relu(x)
        # x = self.pool2(x)

        x = x.view(x.size(0), -1)

        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x

### 🧬 SIR-Based Label Generation

**Definition:**
The SIR (Susceptible–Infected–Recovered) model is used to generate ground truth labels representing node influence.

---

### ⚙️ Computation:

1.  **Epidemic Threshold:** The threshold is calculated as:
    $$\beta_c = \frac{\langle k \rangle}{\langle k^2 \rangle - \langle k \rangle}$$

2.  **Infection Probability:** The probability is set relative to the threshold:
    $$\beta = 1.5\beta_c$$

3.  **Simulation Process:**
    *   Each node is treated as the initial infected node.
    *   The SIR process is simulated multiple times (e.g., 500 runs).
    *   The average number of recovered nodes is calculated as the influence score.

---

### ✅ Normalization:
The labels are normalized to the range $[0, 1]$ to ensure stable model training.

**Interpretation:**
Nodes that infect a larger portion of the network in the SIR simulation are considered more influential and receive higher ground truth scores.

In [18]:
import numpy as np
import time
import os

# Import our optimized memory-efficient helpers
import sir_helpers

# ---- Degree calculations ----
deg = np.array([d for n, d in G.degree()])

k_avg = np.mean(deg)
k2_avg = np.mean(deg**2)

beta_c = k_avg / (k2_avg - k_avg)
beta = 1.5 * beta_c
mu = 1

# ---- Sequential Label Generation with Progress ----
runs = 500
nodelist = list(G.nodes())
dataset_path = os.path.abspath(os.path.join("..", "Datasets", "out.foldoc"))

# Initialize the adjacency list once in the local session memory (takes 0.05 seconds)
sir_helpers.init_worker_from_file(dataset_path)

labels = []
t_start = time.time()

print(f"Starting optimized sequential SIR simulation for {len(nodelist)} nodes...")

for idx, node in enumerate(nodelist):
    spread = 0
    # Run 500 simulations using the optimized function
    for _ in range(runs):
        spread += sir_helpers.SIR_simulation_opt(node, beta, mu)
    
    labels.append(spread / runs)
    
    # Print progress every 500 nodes to keep it smooth
    if (idx + 1) % 500 == 0 or (idx + 1) == len(nodelist):
        elapsed = time.time() - t_start
        percentage = (idx + 1) / len(nodelist) * 100
        est_total = (elapsed / (idx + 1)) * len(nodelist)
        remaining = est_total - elapsed
        print(f"Progress: {idx + 1}/{len(nodelist)} nodes ({percentage:.1f}%) | "
              f"Elapsed: {elapsed:.1f}s | "
              f"Remaining: {remaining:.1f}s | "
              f"Est. Total: {est_total:.1f}s")

labels = np.array(labels)
labels = labels / np.max(labels)

print("\nLabels generated successfully!")
print("Labels:", labels)


Starting optimized sequential SIR simulation for 13356 nodes...
Progress: 500/13356 nodes (3.7%) | Elapsed: 585.5s | Remaining: 15054.4s | Est. Total: 15639.9s
Progress: 1000/13356 nodes (7.5%) | Elapsed: 1086.3s | Remaining: 13422.4s | Est. Total: 14508.7s
Progress: 1500/13356 nodes (11.2%) | Elapsed: 1590.6s | Remaining: 12571.8s | Est. Total: 14162.3s
Progress: 2000/13356 nodes (15.0%) | Elapsed: 2044.5s | Remaining: 11608.9s | Est. Total: 13653.4s
Progress: 2500/13356 nodes (18.7%) | Elapsed: 2464.1s | Remaining: 10700.3s | Est. Total: 13164.4s
Progress: 3000/13356 nodes (22.5%) | Elapsed: 2878.9s | Remaining: 9938.0s | Est. Total: 12816.9s
Progress: 3500/13356 nodes (26.2%) | Elapsed: 3220.1s | Remaining: 9067.7s | Est. Total: 12287.8s
Progress: 4000/13356 nodes (29.9%) | Elapsed: 3534.0s | Remaining: 8266.1s | Est. Total: 11800.1s
Progress: 4500/13356 nodes (33.7%) | Elapsed: 3824.8s | Remaining: 7527.2s | Est. Total: 11351.9s
Progress: 5000/13356 nodes (37.4%) | Elapsed: 4091.1s

In [19]:
import torch
import torch.nn as nn

# ---- Convert data to tensors ----
X = torch.tensor(channels, dtype=torch.float32)

# ---- Normalize input channels per channel ----
X = X - X.mean(dim=(0, 2, 3), keepdim=True)
X = X / (X.std(dim=(0, 2, 3), keepdim=True) + 1e-6)

# ---- Normalize labels for stable regression training ----
y = torch.tensor(labels, dtype=torch.float32).view(-1, 1)
y_mean = y.mean()
y_std = y.std()
y = (y - y_mean) / (y_std + 1e-6)

print("X mean per channel:", X.mean(dim=(0, 2, 3)))
print("X std per channel:", X.std(dim=(0, 2, 3)))
print("y mean:", y_mean.item(), "y std:", y_std.item())

# ---- Initialize model ----
model = NLGCN()

# ---- Optimizer ----
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# ---- Loss function ----
criterion = nn.MSELoss()

# ---- Training Loop ----
epochs = 300

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(X)
    loss = criterion(outputs, y)
    loss.backward()
    optimizer.step()
    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss = {loss.item():.6f}")

# ---- Final predictions ----
model.eval()
with torch.no_grad():
    predictions = model(X)

print("\nFinal Predictions:\n", predictions)

X mean per channel: tensor([ 1.3041e-07, -2.8281e-07, -5.1376e-08, -9.1139e-09, -2.4545e-07,
         4.8916e-07])
X std per channel: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])
y mean: 0.16822242736816406 y std: 0.11975017935037613
Epoch 0, Loss = 1.283063
Epoch 20, Loss = 0.247322
Epoch 40, Loss = 0.186590
Epoch 60, Loss = 0.158871
Epoch 80, Loss = 0.142596
Epoch 100, Loss = 0.131275
Epoch 120, Loss = 0.121923
Epoch 140, Loss = 0.114105
Epoch 160, Loss = 0.106991
Epoch 180, Loss = 0.100360
Epoch 200, Loss = 0.094513
Epoch 220, Loss = 0.089506
Epoch 240, Loss = 0.085173
Epoch 260, Loss = 0.081316
Epoch 280, Loss = 0.077775

Final Predictions:
 tensor([[ 5.5900],
        [ 6.4171],
        [ 6.2874],
        ...,
        [-0.1039],
        [-0.5123],
        [-0.5123]])


### 🔹 Prediction and Ranking Evaluation

**Definition:**
After training, the model predicts influence scores for each node, which are used to rank nodes based on their importance.

---

### ⚙️ Computation:
1.  **Generate Predicted Scores:** The trained model is used to compute influence scores for all nodes in the graph.
2.  **Predicted Ranking:** Nodes are ranked in descending order based on these predicted scores.
3.  **Ground Truth Ranking:** A reference ranking is obtained from the SIR-based simulation labels.
4.  **Comparison:** The predicted ranking is compared with the SIR ranking to measure alignment.

---

### 📊 Evaluation:
*   **Top-k Comparison:** The top-ranked nodes from both predicted and ground truth sets are compared to assess how well the model identifies the most influential nodes.
*   **Ranking Correlation:** Statistical measures can be used to determine the accuracy of the overall node order.

**Key Insight:**
The closer the predicted ranking is to the SIR ranking, the better the model captures the underlying dynamics of node influence within the network.

In [20]:
import numpy as np
import torch

# ---- Prediction ----
model.eval()

with torch.no_grad():
    pred = model(X).detach().cpu().numpy().flatten()

print("Predicted scores:", pred)


# ---- Ranking ----
ranking_pred = np.argsort(pred)[::-1]
ranking_true = np.argsort(labels)[::-1]

print("\nTop predicted nodes:", ranking_pred)
print("Top SIR nodes:", ranking_true)

# Top-k comparison
k = 10
print(f"\nTop {k} predicted nodes:", ranking_pred[:k])
print(f"Top {k} SIR nodes:", ranking_true[:k])

Predicted scores: [ 5.5900245   6.4170876   6.2874107  ... -0.10391432 -0.51232725
 -0.51232725]

Top predicted nodes: [ 567  119  973 ... 7706 7705 7704]
Top SIR nodes: [  119   201   567 ... 11990 12143 11992]

Top 10 predicted nodes: [ 567  119  973  201  285  290   50  168 1225 1341]
Top 10 SIR nodes: [ 119  201  567   50  973  168  285  290 1225 1341]


###  Model Evaluation

**Kendall Tau Correlation:**
The Kendall Tau coefficient is used to measure the similarity between the predicted node ranking and the SIR-based ground truth ranking. A higher value indicates better agreement between the two rankings.

**Top-N Influence Spread:**
The top-N nodes predicted by the model are selected, and their spreading capability is evaluated using the SIR model. The total number of infected nodes represents the effectiveness of the selected nodes.

---

### ✅ Key Insight:
*   **Kendall Tau:** Evaluates the overall ranking consistency.
*   **Top-N Spread:** Evaluates the practical influence performance of the predicted top nodes.

In [22]:
from scipy.stats import kendalltau
# Import the optimized simulation
from sir_helpers import SIR_simulation_opt

# ---- Kendall Tau Correlation ----
tau, p = kendalltau(pred, labels)

print("Kendall Tau:", tau)


# ---- Top-N Influence Spread ----
N = 3   # for small graph (you can change)

top_node_indices = ranking_pred[:N]
top_nodes = [nodelist[idx] for idx in top_node_indices]

# We no longer need to reconstruct 'adj' in this cell because the helper has it loaded globally!
spread_total = 0

for node in top_nodes:
    # ✅ FIX: Removed the 'adj' argument!
    spread_total += SIR_simulation_opt(node, beta, mu)

print("Top-N node indices:", top_node_indices)
print("Top-N nodes:", top_nodes)
print("Spread ability:", spread_total)



Kendall Tau: 0.8134705098544414
Top-N node indices: [567 119 973]
Top-N nodes: [np.int64(6027), np.int64(12404), np.int64(9558)]
Spread ability: 6413


In [23]:
import networkx as nx
from scipy.stats import kendalltau

# Create a copy of G without self-loops for traditional centrality calculations
G_clean = G.copy()
G_clean.remove_edges_from(nx.selfloop_edges(G_clean))

print('Calculating traditional centrality measures for budapest...')

# Degree Centrality
deg_dict = nx.degree_centrality(G_clean)
deg_cent = np.array([deg_dict[n] for n in nodelist])
tau_deg, _ = kendalltau(pred, deg_cent)
print(f'Kendall Tau (Prediction vs Degree): {tau_deg:.4f}')

# Betweenness Centrality
bet_dict = nx.betweenness_centrality(G_clean)
bet_cent = np.array([bet_dict[n] for n in nodelist])
tau_bet, _ = kendalltau(pred, bet_cent)
print(f'Kendall Tau (Prediction vs Betweenness): {tau_bet:.4f}')

# Closeness Centrality
clos_dict = nx.closeness_centrality(G_clean)
clos_cent = np.array([clos_dict[n] for n in nodelist])
tau_clos, _ = kendalltau(pred, clos_cent)
print(f'Kendall Tau (Prediction vs Closeness): {tau_clos:.4f}')

# PageRank
pr_dict = nx.pagerank(G_clean)
pr_cent = np.array([pr_dict[n] for n in nodelist])
tau_pr, _ = kendalltau(pred, pr_cent)
print(f'Kendall Tau (Prediction vs PageRank): {tau_pr:.4f}')

# Coreness (k-core)
core_dict = nx.core_number(G_clean)
core_cent = np.array([core_dict[n] for n in nodelist])
tau_core, _ = kendalltau(pred, core_cent)
print(f'Kendall Tau (Prediction vs Coreness): {tau_core:.4f}')

# Eigenvector Centrality
try:
    eig_dict = nx.eigenvector_centrality(G_clean, max_iter=1000)
    eig_cent = np.array([eig_dict[n] for n in nodelist])
    tau_eig, _ = kendalltau(pred, eig_cent)
    print(f'Kendall Tau (Prediction vs Eigenvector): {tau_eig:.4f}')
except Exception as e:
    print(f'Eigenvector centrality failed: {e}')


Calculating traditional centrality measures for budapest...
Kendall Tau (Prediction vs Degree): 0.6532
Kendall Tau (Prediction vs Betweenness): 0.5391
Kendall Tau (Prediction vs Closeness): 0.6953
Kendall Tau (Prediction vs PageRank): 0.5100
Kendall Tau (Prediction vs Coreness): 0.6698
Kendall Tau (Prediction vs Eigenvector): 0.7308


In [25]:
import os
import numpy as np
import networkx as nx

# Define path to the dataset being used in this notebook
path = os.path.join("..", "Datasets", "out.foldoc")

# Load columns as strings
raw_str = np.loadtxt(path, dtype=str)

if raw_str.ndim == 1:
    raw_str = raw_str.reshape(1, -1)

# Build the weighted graph
G_weighted = nx.Graph()
for row in raw_str:
    u = int(row[0].replace('V', ''))
    v = int(row[1].replace('V', ''))
    
    # If the file has a 3rd column for weights, use it. Otherwise, default to weight=1.0.
    w = float(row[2]) if len(row) >= 3 else 1.0

    # Safety: avoid zero or negative weights
    w = abs(w) if w != 0 else 1e-6

    # Keep the strongest link if edge already exists
    if G_weighted.has_edge(u, v):
        existing = G_weighted[u][v]['weight']
        G_weighted[u][v]['weight'] = max(existing, w)
    else:
        G_weighted.add_edge(u, v, weight=w)

G_weighted.remove_edges_from(nx.selfloop_edges(G_weighted))

print(f"Weighted graph defined successfully! Nodes: {G_weighted.number_of_nodes()}, Edges: {G_weighted.number_of_edges()}")


Weighted graph defined successfully! Nodes: 13356, Edges: 91471


In [26]:
# ---- Weighted Centrality Measures ----
print("\nCalculating weighted centrality measures (positive weight semantics)...")

# Weighted Degree (Strength) — high weight friends boost your score
strength_dict = dict(G_weighted.degree(weight='weight'))
strength_cent = np.array([strength_dict.get(n, 0.0) for n in nodelist])
strength_cent = strength_cent / strength_cent.max() if strength_cent.max() > 0 else strength_cent
tau_wdeg, _ = kendalltau(pred, strength_cent)
print(f"Kendall Tau (Prediction vs Weighted Degree / Strength): {tau_wdeg:.4f}")

# Weighted Betweenness — ✅ distance = 1/weight so high-weight edges are SHORT (preferred paths)
# Strong friendship = easy path = shorter distance
G_weighted_dist = nx.Graph()
G_weighted_dist.add_nodes_from(G_weighted.nodes()) # Initialize with all nodes so none are missed!

for u, v, d in G_weighted.edges(data=True):
    G_weighted_dist.add_edge(u, v, distance=1.0 / d['weight'])  # flip only for path length

wbet_dict = nx.betweenness_centrality(G_weighted_dist, weight='distance', normalized=True)
wbet_cent = np.array([wbet_dict.get(n, 0.0) for n in nodelist]) # Safe .get lookup
tau_wbet, _ = kendalltau(pred, wbet_cent)
print(f"Kendall Tau (Prediction vs Weighted Betweenness):       {tau_wbet:.4f}")

# Weighted Closeness — same distance trick: strong links = short distance
wclos_dict = nx.closeness_centrality(G_weighted_dist, distance='distance')
wclos_cent = np.array([wclos_dict.get(n, 0.0) for n in nodelist]) # Safe .get lookup
tau_wclos, _ = kendalltau(pred, wclos_cent)
print(f"Kendall Tau (Prediction vs Weighted Closeness):         {tau_wclos:.4f}")

# Weighted PageRank — high weight = more rank flows through that edge (direct use)
wpr_dict = nx.pagerank(G_weighted, weight='weight')
wpr_cent = np.array([wpr_dict.get(n, 0.0) for n in nodelist]) # Safe .get lookup
tau_wpr, _ = kendalltau(pred, wpr_cent)
print(f"Kendall Tau (Prediction vs Weighted PageRank):          {tau_wpr:.4f}")

# Weighted Eigenvector — high weight neighbors contribute MORE to your score
try:
    weig_dict = nx.eigenvector_centrality(G_weighted, weight='weight', max_iter=1000)
    weig_cent = np.array([weig_dict.get(n, 0.0) for n in nodelist]) # Safe .get lookup
    tau_weig, _ = kendalltau(pred, weig_cent)
    print(f"Kendall Tau (Prediction vs Weighted Eigenvector):       {tau_weig:.4f}")
except nx.PowerIterationFailedConvergence:
    print("Weighted Eigenvector did not converge — trying numpy fallback...")
    weig_dict = nx.eigenvector_centrality_numpy(G_weighted, weight='weight')
    weig_cent = np.array([weig_dict.get(n, 0.0) for n in nodelist]) # Safe .get lookup
    tau_weig, _ = kendalltau(pred, weig_cent)
    print(f"Kendall Tau (Prediction vs Weighted Eigenvector):       {tau_weig:.4f}")



Calculating weighted centrality measures (positive weight semantics)...
Kendall Tau (Prediction vs Weighted Degree / Strength): 0.6538
Kendall Tau (Prediction vs Weighted Betweenness):       0.4458
Kendall Tau (Prediction vs Weighted Closeness):         0.6103
Kendall Tau (Prediction vs Weighted PageRank):          0.5169
Kendall Tau (Prediction vs Weighted Eigenvector):       0.7024
